# 🔁 第 5 阶段：Agent Loop（核心突破）

## 🎯 阶段目标

实现真正的 Agent Loop：

> **思考 → 行动 → 观察 → 再思考**

这是从"单步工具调用"到"真正 Agent"的分水岭。

---

## 🧠 核心概念

Agent Loop 流程图：

```
思考 → 行动 → 执行 → 观察 → 再思考
    ↓                      ↑
    └──────────────────────┘
```

---

## 📦 1. 加载配置

In [1]:
import json
import requests
import re
from pathlib import Path

# 加载配置
config_path = Path.cwd() / ".." / "config.json"
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_CONFIG = config["model"]

print("✅ 配置加载成功")

✅ 配置加载成功


---

## 🔧 2. 定义工具

In [2]:
# 工具描述
tools = [
    {
        "name": "get_weather",
        "description": "获取指定城市的天气信息",
        "parameters": {
            "city": {"type": "string", "description": "城市名称"}
        }
    },
    {
        "name": "calculate",
        "description": "进行数学计算",
        "parameters": {
            "expression": {"type": "string", "description": "数学表达式"}
        }
    },
    {
        "name": "get_time",
        "description": "获取当前时间",
        "parameters": {}
    }
]

# 工具实现
def get_weather(city: str) -> str:
    weather_database = {
        "北京": {"天气": "晴天", "温度": "25°C"},
        "上海": {"天气": "多云", "温度": "23°C"},
        "广州": {"天气": "小雨", "温度": "28°C"}
    }
    if city in weather_database:
        data = weather_database[city]
        return f"{city}的天气：{data['天气']}，温度{data['温度']}"
    return f"未知城市: {city}"

def calculate(expression: str) -> str:
    try:
        allowed_chars = "0123456789+-*/(). "
        if all(c in allowed_chars for c in expression):
            return f"{expression} = {eval(expression)}"
        return "非法表达式"
    except:
        return "计算错误"

def get_time() -> str:
    from datetime import datetime
    return datetime.now().strftime("%Y年%m月%d日 %H:%M:%S")

tool_map = {
    "get_weather": get_weather,
    "calculate": calculate,
    "get_time": get_time
}

print("✅ 工具定义完成")

✅ 工具定义完成


---

## 🚀 3. 定义 LLM 调用和解析函数

In [3]:
def call_llm(messages: list) -> str:
    url = f"{MODEL_CONFIG['base_url']}/v1/chat/completions"
    payload = {
        "model": MODEL_CONFIG['name'],
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": 200
    }
    try:
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"❌ 错误：{str(e)}"

def extract_json(response: str) -> dict:
    try:
        return json.loads(response)
    except:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                return None
        return None

print("✅ LLM 调用和解析函数定义完成")

✅ LLM 调用和解析函数定义完成


---

## 🎯 4. Agent Loop 核心逻辑

In [4]:
def run_agent_loop(user_input: str, max_steps: int = 5) -> str:
    """
    运行完整的 Agent Loop
    """
    history = []
    
    tools_description = "\n".join([
        f"{i+1}. {tool['name']}: {tool['description']}"
        for i, tool in enumerate(tools)
    ])
    
    system_prompt = f"""
你是一个智能助手，可以使用工具来回答问题。

可用工具：
{tools_description}

请输出结构化的 JSON 响应：
{{
  "thought": "你的思考过程",
  "action": "工具名称或 'finish'",
  "params": {{参数对象}}
}}

注意：
- 如果已经有足够信息回答用户问题，action 填 "finish"
- 如果需要继续调用工具获取更多信息，action 填工具名称
"""
    
    print(f"🎯 用户问题: {user_input}")
    print("=" * 60)
    
    for step in range(max_steps):
        print(f"\n📋 步骤 {step + 1}/{max_steps}")
        
        # 构建历史摘要
        history_summary = "\n".join([
            f"步骤 {i+1}: {h['action']} -> {h['observation']}"
            for i, h in enumerate(history)
        ])
        
        # 生成提示
        user_prompt = f"""
用户问题：{user_input}

历史记录：
{history_summary if history_summary else "无"}

请决定下一步行动。
"""
        
        # 思考并决定
        messages = [
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_prompt.strip()}
        ]
        
        response = call_llm(messages)
        action = extract_json(response)
        
        if not action:
            return f"步骤 {step+1}: 无法解析模型响应"
        
        thought = action.get("thought", "")
        action_name = action.get("action", "")
        params = action.get("params", {})
        
        print(f"🤔 思考: {thought}")
        print(f"⚡ 行动: {action_name}")
        
        # 检查终止条件
        if action_name == "finish":
            print("🏁 循环结束")
            final_prompt = f"""
根据以下对话历史，总结回答用户问题：

用户问题：{user_input}

对话历史：
{history_summary if history_summary else "无"}

请用自然语言给出最终回答。
"""
            return call_llm([{"role": "user", "content": final_prompt}])
        
        # 执行工具
        if action_name not in tool_map:
            observation = f"未知工具: {action_name}"
        else:
            try:
                tool_func = tool_map[action_name]
                observation = tool_func(**params)
            except Exception as e:
                observation = f"工具调用失败: {str(e)}"
        
        print(f"🔍 观察: {observation}")
        
        # 记录状态
        history.append({
            "thought": thought,
            "action": action_name,
            "observation": observation
        })
    
    return f"⚠️ 已达到最大步数 ({max_steps})，任务未完成"

print("✅ Agent Loop 定义完成")

✅ Agent Loop 定义完成


---

## 🎮 5. 测试 Agent Loop

In [5]:
# 测试 1：单步任务
print("\n=== 测试 1：单步任务 ===")
answer = run_agent_loop("北京今天天气怎么样？")
print(f"\n🎉 最终回答：{answer}")


=== 测试 1：单步任务 ===
🎯 用户问题: 北京今天天气怎么样？

📋 步骤 1/5
🤔 思考: 用户询问北京今天的天气情况，我需要使用 get_weather 工具来获取北京的天气信息。
⚡ 行动: get_weather
🔍 观察: 北京的天气：晴天，温度25°C

📋 步骤 2/5
🤔 思考: 根据历史记录，步骤1已经获取了北京的天气信息：晴天，温度25°C。现在已经有足够的信息回答用户的问题，不需要再调用其他工具。
⚡ 行动: finish
🏁 循环结束

🎉 最终回答：Thinking Process:

1.  **Analyze the Request:**
    *   Input: User question ("北京今天天气怎么样？" - How's the weather in Beijing today?) and Dialogue History (Step 1: get_weather -> Beijing weather: Sunny, 25°C).
    *   Task: Summarize and answer the user's question based on the dialogue history.
    *   Output Format: Natural language response.

2.  **Analyze the Dialogue History:**
    *   The history shows a tool call/result: `get_weather` was executed.
    *   The result is: 北京的天气：晴天，温度 25°C (Beijing weather: Sunny, Temperature 25°C).

3.  **Formulate the Answer:**
    *   The user asked about Beijing's weather today.
    *   The retrieved information says it's sunny and 25°C.
    *   I need to combine


In [6]:
# 测试 2：多步任务
print("\n=== 测试 2：多步任务 ===")
answer = run_agent_loop("先查北京天气，再查上海天气，然后对比一下")
print(f"\n🎉 最终回答：{answer}")


=== 测试 2：多步任务 ===
🎯 用户问题: 先查北京天气，再查上海天气，然后对比一下

📋 步骤 1/5
🤔 思考: 用户需要先查北京天气，再查上海天气，然后对比。目前没有任何历史记录，我需要先获取北京的天气信息。
⚡ 行动: get_weather
🔍 观察: 北京的天气：晴天，温度25°C

📋 步骤 2/5
🤔 思考: 用户要求先查北京天气，再查上海天气，然后对比。历史记录显示步骤1已经获取了北京的天气信息（晴天，25°C）。现在需要获取上海的天气信息，才能进行对比。
⚡ 行动: get_weather
🔍 观察: 上海的天气：多云，温度23°C

📋 步骤 3/5

🎉 最终回答：步骤 3: 无法解析模型响应


---

## 🚨 重要里程碑

> **🎉 恭喜！到这里，你已经实现了真正的 Agent！**

你的 Agent 具备：
- ✅ 自主思考能力
- ✅ 工具调用能力
- ✅ 循环迭代能力
- ✅ 结果总结能力

---

## ✅ 本阶段总结

你已经完成了：

1. ✅ 实现了完整的 Agent Loop
2. ✅ 理解了循环控制机制
3. ✅ 学会了中间状态记录

---

## 🔗 下一步

进入 **[第 6 阶段：模型做计划](../step6/step6.ipynb)**，学习如何让 Agent 制定和执行计划。